# Step 5: Sampling and LLM-as-Judge

This notebook samples proposals for human/LLM evaluation (5-point Likert scale), runs the LLM-as-judge scoring, and computes inter-annotator agreement.

In [ ]:
# © Copyright European Union - 2026

import os
import json
import re
import random
import glob
import pandas as pd
import numpy as np
from collections import Counter
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.environ["LLM_BASE_URL"]
)

MODEL_NAME = "gpt-oss-120b"

random.seed(42)


def extract_json_from_reply(raw_output: str):
    """Extract JSON from LLM reply text."""
    try:
        return json.loads(raw_output)
    except (json.JSONDecodeError, TypeError):
        pass
    raw_output = raw_output.strip()
    if "```json" in raw_output:
        raw_output = raw_output.split("```json")[-1].split("```")[0]
    elif "```" in raw_output:
        parts = raw_output.split("```")
        if len(parts) >= 3:
            raw_output = parts[1]
    # Find JSON structure
    for start_char, end_char in [("[", "]"), ("{", "}")]:
        if start_char in raw_output and end_char in raw_output:
            start = raw_output.index(start_char)
            end = raw_output.rindex(end_char) + 1
            try:
                return json.loads(raw_output[start:end])
            except json.JSONDecodeError:
                continue
    return json.loads(raw_output)


def model_json(system_prompt: str, user_message: str) -> dict:
    """Call LLM and return parsed JSON response."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": str(user_message)}
        ],
        temperature=0
    )
    return extract_json_from_reply(response.choices[0].message.content)


def model(system_prompt: str, user_message: str) -> str:
    """Call LLM and return raw text response."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": str(user_message)}
        ],
        temperature=0
    )
    return response.choices[0].message.content


In [ ]:
DATA_OUT_BASE_PATH = "../out/"
clusterized_proposals_fp = f"{DATA_OUT_BASE_PATH}3.clusterized_proposals.json"
filtered_clusters_fp = f"{DATA_OUT_BASE_PATH}5.filtered_clusters.json"

full_summaries_fp = f"{DATA_OUT_BASE_PATH}5.full_cluster_summaries.json"
filtered_summaries_fp = f"{DATA_OUT_BASE_PATH}5.filtered_cluster_summaries.json"

out_full_samples_fp = f"{DATA_OUT_BASE_PATH}6.full_samples.json"
out_clean_samples_fp = f"{DATA_OUT_BASE_PATH}6.samples.json"
out_clean_samples_csv_fp = f"{DATA_OUT_BASE_PATH}6.samples.xlsx"

In [ ]:
OUT_OF_CLUSTER_MAX_NUMBER = 5
SAMPLE_SIZE = 10
SEED = 42
LANGUAGES = ['it', 'en', 'fr']
random.seed(SEED)

## Sample Data first level

In [ ]:
def sample_data(from_list:list, number:int, is_fixed_limit:bool=True):
    random.shuffle(from_list)
    sample_size = number
    if not is_fixed_limit:
        sample_size = random.sample(range(1, number+1), 1)[0]
    sample = random.sample(from_list, sample_size)
    return sample

In [ ]:
clusterized_proposals = json.load(open(clusterized_proposals_fp))
cluster_summaries = json.load(open(full_summaries_fp))

In [ ]:
cluster_samples_clean = {}
cluster_samples_full = {}
annotation_lines = []
for cluster_name, cluster_proposals in clusterized_proposals.items():
    cluster_llm_title = cluster_summaries[cluster_name]['title']
    cluster_llm_description = cluster_summaries[cluster_name]['description']
    print(cluster_name)
    print("\t", cluster_llm_title)
    print("\t", cluster_llm_description)

    lang_cluster_samples = {lang: [x for x in clusterized_proposals[cluster_name] if x['language'] == lang] for lang in LANGUAGES}
    lang_non_cluster_samples = {lang: [{'cluster': other_cluster_name, 'item': x} for other_cluster_name in clusterized_proposals.keys() if other_cluster_name != cluster_name for x in clusterized_proposals[other_cluster_name] if x['language'] == lang] for lang in LANGUAGES}

    language_data_clean = {}
    language_data_full = {}
    for language in LANGUAGES:
        clean_list = []
        full_list = []

        non_cluster_items = sample_data(lang_non_cluster_samples[language], OUT_OF_CLUSTER_MAX_NUMBER, is_fixed_limit=False)
        cluster_items = sample_data(lang_cluster_samples[language], (SAMPLE_SIZE-len(non_cluster_items)))
        full_list = [{'cluster': cluster_name, 'item': x} for x in cluster_items] + non_cluster_items
        clean_list = cluster_items + [x['item'] for x in non_cluster_items]
        random.shuffle(clean_list)
        clean_list = [{k:v for k,v in x.items() if k != "UGC"} for x in clean_list]

        print(language)
        print("Non cluster elements", len(non_cluster_items))
        print("Cluster elements", len(cluster_items))

        language_data_clean[language] = clean_list
        language_data_full[language] = full_list

        for element in clean_list:
            line = {'llm_title': cluster_llm_title,'llm_description': cluster_llm_description}
            line.update(element)
            line["Q1"] = ""
            line["Q2"] = ""
            line["Q3"] = ""
            annotation_lines.append(line)

    cluster_samples_clean[cluster_name] = {
        'llm_title': cluster_llm_title,
        'llm_description': cluster_llm_description,
        'samples': language_data_clean
    }

    cluster_samples_full[cluster_name] = {
        'llm_title': cluster_llm_title,
        'llm_description': cluster_llm_description,
        'samples': language_data_full
    }

json.dump(cluster_samples_clean, open(out_clean_samples_fp, 'w'), indent=2, ensure_ascii=False)
json.dump(cluster_samples_full, open(out_full_samples_fp, 'w'), indent=2, ensure_ascii=False)
pd.DataFrame(annotation_lines).to_excel(out_clean_samples_csv_fp, index=False)

In [ ]:
df = pd.read_excel(out_clean_samples_csv_fp)
full_cluster_samples = json.load(open(out_full_samples_fp))
full_cluster_summaries = json.load(open(f"{DATA_OUT_BASE_PATH}5.full_cluster_summaries.json"))

belong_to_cluster = []
original_cluster = []
llm_title_to_real_title = {v['title']:k for k,v in full_cluster_summaries.items()}
for idx, row in df.iterrows():
    sample_annotation_cluster = row['llm_title']
    sample_annotation_cluster_real = llm_title_to_real_title[row['llm_title']]
    full_sample = [sample for lang,lang_samples in full_cluster_samples[sample_annotation_cluster_real]['samples'].items() for sample in lang_samples if sample['item']['id'] == row['id']][0]

    if full_sample['cluster'] == sample_annotation_cluster_real:
        belong_to_cluster.append(True)
    else:
        belong_to_cluster.append(False)
    original_cluster.append(full_sample['cluster'])

df['Belong to cluster'] = belong_to_cluster
df['Original to cluster'] = original_cluster
df.to_excel(out_clean_samples_csv_fp)

### Average scores first level

In [ ]:
# load data
df = pd.read_excel(f"{DATA_OUT_BASE_PATH}6.samples_llm-judge-elisa-nicolas.xlsx")

for annotator in ['llm', 'Elisa', 'Nicolas']:
    score_column = annotator if annotator != 'llm' else 'Q3'
    scores_in_cluster = {}
    scores_out_cluster = {}

    scores_in_cluster_lang = {}
    scores_out_cluster_lang = {}

    in_clusters = []
    out_clusters = []

    fr_in_clusters = []
    it_in_clusters = []
    en_in_clusters = []
    fr_out_clusters = []
    it_out_clusters = []
    en_out_clusters = []
    for idx, row in df.iterrows():
        cluster_title = row['llm_title']
        language = row['language']

        #if annotator != 'Elisa' or language != 'fr':
        if cluster_title not in scores_in_cluster:
            scores_in_cluster[cluster_title] = []
        if cluster_title not in scores_out_cluster:
            scores_out_cluster[cluster_title] = []

        # Lang
        if cluster_title not in scores_in_cluster_lang:
            scores_in_cluster_lang[cluster_title] = {}
        if language not in scores_in_cluster_lang[cluster_title]:
            scores_in_cluster_lang[cluster_title][language] = []

        if cluster_title not in scores_out_cluster_lang:
            scores_out_cluster_lang[cluster_title] = {}
        if language not in scores_out_cluster_lang[cluster_title]:
            scores_out_cluster_lang[cluster_title][language] = []

        if row['Belong to cluster']:
            scores_in_cluster[cluster_title].append(row[score_column])
            scores_in_cluster_lang[cluster_title][language].append(row[score_column])
            in_clusters.append(row[score_column])
            if language == 'fr':
                fr_in_clusters.append(row[score_column])
            if language == 'it':
                it_in_clusters.append(row[score_column])
            if language == 'en':
                en_in_clusters.append(row[score_column])
        else:
            scores_out_cluster[cluster_title].append(row[score_column])
            scores_out_cluster_lang[cluster_title][language].append(row[score_column])
            out_clusters.append(row[score_column])
            if language == 'fr':
                fr_out_clusters.append(row[score_column])
            if language == 'it':
                it_out_clusters.append(row[score_column])
            if language == 'en':
                en_out_clusters.append(row[score_column])
        # End if
    # End for
    results = []
    for cluster_title in scores_in_cluster:
        results.append({
            'cluster_name': cluster_title,
            'AVG (in)': np.mean(scores_in_cluster[cluster_title]),
            'Support (in)': len(scores_in_cluster[cluster_title]),
            'AVG (out)': np.mean(scores_out_cluster[cluster_title]),
            'Support (out)': len(scores_out_cluster[cluster_title]),
            'FR (in)': np.mean(scores_in_cluster_lang[cluster_title]['fr']) if scores_in_cluster_lang[cluster_title].get('fr') else 0,
            'FR (out)': np.mean(scores_out_cluster_lang[cluster_title]['fr']) if scores_out_cluster_lang[cluster_title].get('fr') else 0,
            'IT (in)': np.mean(scores_in_cluster_lang[cluster_title]['it']),
            'IT (out)': np.mean(scores_out_cluster_lang[cluster_title]['it']),
            'EN (in)': np.mean(scores_in_cluster_lang[cluster_title]['en']),
            'EN (out)': np.mean(scores_out_cluster_lang[cluster_title]['en']),
        })
    results.append({
            'cluster_name': "Overall",
            'AVG (in)': np.mean(in_clusters),
            'Support (in)': len(in_clusters),
            'AVG (out)': np.mean(out_clusters),
            'Support (out)': len(out_clusters),
            'FR (in)': np.mean(fr_in_clusters) if fr_in_clusters else 0,
            'FR (out)': np.mean(fr_out_clusters) if fr_out_clusters else 0,
            'IT (in)': np.mean(it_in_clusters),
            'IT (out)': np.mean(it_out_clusters),
            'EN (in)': np.mean(en_in_clusters),
            'EN (out)': np.mean(en_out_clusters),
        })

    result_df = pd.DataFrame(results)
    result_df.to_excel(f"{DATA_OUT_BASE_PATH}6.anaysis_results_{annotator}.xlsx", index=False)

## Sample Data second level

In [ ]:
DATA_OUT_BASE_PATH = "../out/"
clusterized_proposals_fp = f"{DATA_OUT_BASE_PATH}7.second_level_clusterized_proposals.json"
cluster_summaries_fp = f"{DATA_OUT_BASE_PATH}7.cluster_subcluster.json"

out_clean_samples_csv_fp = f"{DATA_OUT_BASE_PATH}6.second_level_samples.xlsx"

In [ ]:
def sample_data(from_list:list, number:int, is_fixed_limit:bool=True):
    random.shuffle(from_list)
    sample_size = number
    if not is_fixed_limit:
        sample_size = random.sample(range(1, number+1), 1)[0]
    else:
        if number == len(from_list):
            return from_list
    sample = random.sample(from_list, sample_size)
    return sample

In [ ]:
clusterized_proposals = json.load(open(clusterized_proposals_fp))
cluster_descriptions = json.load(open(cluster_summaries_fp))
sub_cluster_to_items = {sub_cluster: sub_cluster_items  for cluster, sub_cluster_dict in clusterized_proposals.items() for sub_cluster, sub_cluster_items in sub_cluster_dict.items()}
sub_cluster_to_description = {sub_cluster_data['name']: {'first_level_cluster': cluster, 'summary': sub_cluster_data['summary']}  for cluster, sub_cluster_list in cluster_descriptions.items() for sub_cluster_data in sub_cluster_list}

### Query LLM first level

In [ ]:
annotation_lines = []
for cluster_name, cluster_proposals in sub_cluster_to_items.items():
    cluster_llm_title = cluster_name
    cluster_llm_description = sub_cluster_to_description[cluster_name]['summary']
    print(cluster_name)
    print("\t", cluster_llm_title)
    print("\t", cluster_llm_description)

    lang_cluster_samples = {lang: [x for x in sub_cluster_to_items[cluster_name] if x['language'] == lang] for lang in LANGUAGES}
    lang_non_cluster_samples = {lang: [{'cluster': other_cluster_name, 'item': x} for other_cluster_name in sub_cluster_to_items.keys() if other_cluster_name != cluster_name for x in sub_cluster_to_items[other_cluster_name] if x['language'] == lang] for lang in LANGUAGES}

    for language in LANGUAGES:
        out_of_treshold = max(OUT_OF_CLUSTER_MAX_NUMBER, SAMPLE_SIZE-len(lang_cluster_samples[language]))
        non_cluster_items = sample_data(lang_non_cluster_samples[language], out_of_treshold, is_fixed_limit=len(lang_cluster_samples[language])<9)
        print("\tNon cluster elements selected", len(non_cluster_items), "threshold", out_of_treshold)
        print("\tCluster samples", len(lang_cluster_samples[language]))
        print("\tCluster samples we need", SAMPLE_SIZE-len(non_cluster_items))
        cluster_items = sample_data(lang_cluster_samples[language], (SAMPLE_SIZE-len(non_cluster_items)))
        full_list = [{'cluster': cluster_name, 'item': x} for x in cluster_items] + non_cluster_items
        random.shuffle(full_list)

        print(language)
        print("Non cluster elements", len(non_cluster_items))
        print("Cluster elements", len(cluster_items))

        for full_element in full_list:
            original_cluster = full_element['cluster']
            element = full_element['item']
            line = {'llm_title': cluster_llm_title,'llm_description': cluster_llm_description}
            line['title'] = element['title']
            line["Q3"] = ""
            line['id'] = element['id']
            line['language'] = element['language']
            line['Belong to cluster'] = original_cluster == cluster_name
            line['Original to cluster'] = original_cluster
            annotation_lines.append(line)

pd.DataFrame(annotation_lines).to_excel(out_clean_samples_csv_fp, index=False)

### Second level query llm

In [ ]:
from tqdm import tqdm
# load data
df = pd.read_excel(out_clean_samples_csv_fp)

for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
    prompt_cluster = f"""Your task is to decide wether a sentence belong to a group, given the group title and its description.
    Give a score from 1 to 5 to judge if the sentence belongs to the group, where 1 means that the sentence is not belonging to the set, while 5 means that it definitely belongs to it.

    Provide in output a JSON with the following structure, do not output anything else.

    class Score(BaseModel):
        score: int # The score indicating if the sentence belongs to the group identified by title and description. The score can range in the continuum from 1 to 5. Here 1 means that the sentence is not belonging to the set, while 5 means that it definitely belongs to it.

    Output:
        Score # the score assigned to the sentence
        """
    text_prompt = f"""
    Here is the group title:
    {row['llm_title']}
    Here is the group description:
    {row['llm_description']}

    Here is the sentence:
    {row['title']}"""
    r = model_json(prompt_cluster, text_prompt)
    df.at[idx, 'Q3'] = r['score']

df.to_excel(f"{DATA_OUT_BASE_PATH}6.samples_llm-judge.xlsx", index=False)

In [ ]:
from tqdm import tqdm
# load data
df = pd.read_excel(out_clean_samples_csv_fp)

for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
    prompt_cluster = f"""Your task is to decide wether a sentence belong to a group, given the group title and its description.
    Give a score from 1 to 5 to judge if the sentence belongs to the group, where 1 means that the sentence is not belonging to the set, while 5 means that it definitely belongs to it.

    Provide in output a JSON with the following structure, do not output anything else.

    class Score(BaseModel):
        score: int # The score indicating if the sentence belongs to the group identified by title and description. The score can range in the continuum from 1 to 5. Here 1 means that the sentence is not belonging to the set, while 5 means that it definitely belongs to it.

    Output:
        Score # the score assigned to the sentence
        """
    text_prompt = f"""
    Here is the group title:
    {row['llm_title']}
    Here is the group description:
    {row['llm_description']}

    Here is the sentence:
    {row['title']}"""
    r = model_json(prompt_cluster, text_prompt)
    df.at[idx, 'Q3'] = r['score']
    if idx % 200 == 0:
        df.to_excel(f"{DATA_OUT_BASE_PATH}6.second-level_samples_llm-judge.xlsx", index=False)    

df.to_excel(f"{DATA_OUT_BASE_PATH}6.second-level_samples_llm-judge.xlsx", index=False)

### Compute V-measure for second level clustering

In [ ]:
from tqdm import tqdm
df = pd.read_excel(f"{DATA_OUT_BASE_PATH}6.second-level_samples_llm-judge.xlsx")
sub_cluster_to_items = {sub_cluster: sub_cluster_items  for cluster, sub_cluster_dict in clusterized_proposals.items() for sub_cluster, sub_cluster_items in sub_cluster_dict.items()}
sub_clusters = [sub_cluster_data['name'] for cluster, sub_cluster_list in cluster_descriptions.items() for sub_cluster_data in sub_cluster_list]
sub_cluster_to_description = {sub_cluster_data['name']: {'first_level_cluster': cluster, 'summary': sub_cluster_data['summary']}  for cluster, sub_cluster_list in cluster_descriptions.items() for sub_cluster_data in sub_cluster_list}

prompt_classify = """Your task is to find the most appropriate class for a given text.
In what follows you can find the set of classes together with their definition.
You can choose one of the following classes:
%s
Output a JSON with the following structure, do not output anything else.
Before outputin check the JSON is valid. Correct it if it is not

Output:
    class: str # the class corresponding to the given text.
"""
for cluster in sub_cluster_to_description:
    prompt_classify %= (f"- {cluster}: {sub_cluster_to_description[cluster]['summary']}\n%s")
prompt_classify %= ""

label2id = {k:idx for idx,k in enumerate(sub_clusters)}
print(label2id)
labels_true = []
labels_pred = []
labels_pred_txt = []
for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
    score = row['Q3']
    if score < 3:
        # Need to re-annotate data
        text_prompt = f"""
        Here is the text:
        {row['title']}
        """
        r = model_json(prompt_classify, text_prompt)
        print(r)
        if r['class'] in label2id:
            labels_true.append(label2id[row['Original to cluster']])
            labels_pred.append(label2id[r['class']])
            labels_pred_txt.append(r['class'])
    else:
        labels_true.append(label2id[row['Original to cluster']])
        labels_pred.append(label2id[row['llm_title']])
        labels_pred_txt.append(row['llm_title'])


df['Pred. cluster'] = labels_pred_txt + ["None"]* (len(df)-len(labels_pred_txt))
df.to_excel(f"{DATA_OUT_BASE_PATH}6.second-level_samples_llm-judge_additional_preds.xlsx")

# Compute V-Measure
from sklearn.metrics import v_measure_score
v_measure = v_measure_score(labels_pred, labels_true)
print("LLM V-Measure :", v_measure)

### Average scores second level

In [ ]:
# load data

for annotator in ['llm', 'Nicolas']:
    if annotator == 'llm':
        df = pd.read_excel(f"{DATA_OUT_BASE_PATH}6.second-level_samples_llm-judge_additional_preds.xlsx")
    else:
        df = pd.read_excel(f"{DATA_OUT_BASE_PATH}6.second_level_samples_annotations-Nicolas.xlsx")
    score_column = 'Q3'
    scores_in_cluster = {}
    scores_out_cluster = {}

    scores_in_cluster_lang = {}
    scores_out_cluster_lang = {}

    in_clusters = []
    out_clusters = []

    fr_in_clusters = []
    it_in_clusters = []
    en_in_clusters = []
    fr_out_clusters = []
    it_out_clusters = []
    en_out_clusters = []
    for idx, row in df.iterrows():
        cluster_title = row['llm_title']
        language = row['language']

        #if annotator != 'Elisa' or language != 'fr':
        if cluster_title not in scores_in_cluster:
            scores_in_cluster[cluster_title] = []
        if cluster_title not in scores_out_cluster:
            scores_out_cluster[cluster_title] = []

        # Lang
        if cluster_title not in scores_in_cluster_lang:
            scores_in_cluster_lang[cluster_title] = {}
        if language not in scores_in_cluster_lang[cluster_title]:
            scores_in_cluster_lang[cluster_title][language] = []

        if cluster_title not in scores_out_cluster_lang:
            scores_out_cluster_lang[cluster_title] = {}
        if language not in scores_out_cluster_lang[cluster_title]:
            scores_out_cluster_lang[cluster_title][language] = []

        if row['Belong to cluster']:
            scores_in_cluster[cluster_title].append(row[score_column])
            scores_in_cluster_lang[cluster_title][language].append(row[score_column])
            in_clusters.append(row[score_column])
            if language == 'fr':
                fr_in_clusters.append(row[score_column])
            if language == 'it':
                it_in_clusters.append(row[score_column])
            if language == 'en':
                en_in_clusters.append(row[score_column])
        else:
            scores_out_cluster[cluster_title].append(row[score_column])
            scores_out_cluster_lang[cluster_title][language].append(row[score_column])
            out_clusters.append(row[score_column])
            if language == 'fr':
                fr_out_clusters.append(row[score_column])
            if language == 'it':
                it_out_clusters.append(row[score_column])
            if language == 'en':
                en_out_clusters.append(row[score_column])
        # End if
    # End for
    results = []
    for cluster_title in scores_in_cluster:
        results.append({
            'cluster_name': cluster_title,
            'AVG (in)': np.mean(scores_in_cluster[cluster_title]),
            'Support (in)': len(scores_in_cluster[cluster_title]),
            'AVG (out)': np.mean(scores_out_cluster[cluster_title]),
            'Support (out)': len(scores_out_cluster[cluster_title]),
            'FR (in)': np.mean(scores_in_cluster_lang[cluster_title]['fr']) if scores_in_cluster_lang[cluster_title].get('fr') else 0,
            'FR (out)': np.mean(scores_out_cluster_lang[cluster_title]['fr']) if scores_out_cluster_lang[cluster_title].get('fr') else 0,
            'IT (in)': np.mean(scores_in_cluster_lang[cluster_title]['it'] if scores_in_cluster_lang[cluster_title].get('it') else 0),
            'IT (out)': np.mean(scores_out_cluster_lang[cluster_title]['it'] if scores_in_cluster_lang[cluster_title].get('it') else 0),
            'EN (in)': np.mean(scores_in_cluster_lang[cluster_title]['en'] if scores_in_cluster_lang[cluster_title].get('en') else 0),
            'EN (out)': np.mean(scores_out_cluster_lang[cluster_title]['en'] if scores_in_cluster_lang[cluster_title].get('en') else 0),
        })
    results.append({
            'cluster_name': "Overall",
            'AVG (in)': np.mean(in_clusters),
            'Support (in)': len(in_clusters),
            'AVG (out)': np.mean(out_clusters),
            'Support (out)': len(out_clusters),
            'FR (in)': np.mean(fr_in_clusters) if fr_in_clusters else 0,
            'FR (out)': np.mean(fr_out_clusters) if fr_out_clusters else 0,
            'IT (in)': np.mean(it_in_clusters) if it_in_clusters else 0,
            'IT (out)': np.mean(it_out_clusters) if it_out_clusters else 0,
            'EN (in)': np.mean(en_in_clusters) if en_in_clusters else 0,
            'EN (out)': np.mean(en_out_clusters) if en_out_clusters else 0,
        })

    result_df = pd.DataFrame(results)
    result_df.to_excel(f"{DATA_OUT_BASE_PATH}6.anaysis_results_{annotator}_second_level.xlsx", index=False)

In [ ]:
from sklearn.metrics import cohen_kappa_score
llm_data = [4,3,5,4,5,5,5,5,5,4,2,1,5,2,2,5,2,5,4,5,5,4,4,4,5,5,2,2,3,5,3,5,5,5,1,4,1,5,5,4,2,2,2,1,2,2,1,1,4,2,1,1,1,1,1,5,4,3,1,1,5,2,3,2,5,5,4,1,2,4,2,2,5,2,4,1,5,3,1,1,5,4,2,2,1,2,5,4,2,2,1,5,5,1,1,5,4,2,1,1,1,2,2,1,2,3,4,2,5,4,1,1,1,1,1,2,1,1,1,1,1,2,5,1,5,1,2,3,1,1,4,4,4,3,2,3,2,3,4,4,1,1,5,4,4,5,5,5,5,5,3,3,5,1,3,4,1,1,5,5,4,1,1,1,2,5,1,1,4,4,1,1,3,1,1,5,1,2,1,1,5,4,1,1,2,1,1,1,2,1,3,4,4,4,5,4,4,4,2,4,5,1,5,5,5,5,5,4,1,5,2,1,1,4,1,1,1,5,5,5,5,3,4,1,2,5,5,1,5,1,2,2,1,2,1,1,1,1,1,1,5,1,4,4,1,1,1,1,4,5,1,2,1,2,1,1,1,2,1,1,3,5,5,3,5,4,5,1,5,5,1,4,5,2,1,4,5,1,1,1,1,2,4,2,1,5,5,2,5,3,1,2,1,1,1,4,4,1,5,1,1,1,2,1,1,1,5,1,1,2,1,1,1,1,5,3,3,1,4,1,4,1,3,5,2,4,1,1,5,4,3,4,1,1,5,4,5,5,5,5,1,5,2,2,5,2,5,3,5,4,4,1,4,3,1,1,2,5,4,2,2,3,2,5,2,1,2,5,2,1,5,1,5,1,1,1,4,1,1,3,2,3,2,2,1,1,1,1,1,1,3,3,1,1,2,1,5,2,1,3,1,1,5,1,1,1,1,5,1,2,1,1,1,1,4,1,1,1,1,1,1,1,1,3,1,1,1,1,5,2,5,1,3,1,2,4,1,1,3,4,3,2,1,2,1,1,1,1,2,1,1,1,1,1,1,1,1,5,2,1,5,2,5,5,1,4,3,1,5,5,4,5,5,5,5,5,4,5,5,4,5,2,5,2,5,3,4,4,4,1,4,5,5,3,4,5,1,5,5,5,1,1,4,5,5,2,3,5,1,5,1,5,1,1,5,5,5,1,3,2,2,5,5,4,5,1,2,1,1,4,5,4,1,1,1,5,1,2,4,1,5,1,5,5,1,2,5,1,1,2,1,1,1,1,1,1,1,1,1,1,1,5,1,1,2,2,1,1,1,1,5,3,3,5,5,5,1,5,5,5,5,1,1,1,1,1,5,5,5,4,4,1,1,5,5,4,1,1,5,5,1,2,1,4,5,1,1,1,1,3]
rater = [3,5,5,5,4,5,5,5,5,4,0,1,5,3,5,5,5,5,5,5,5,5,5,5,5,5,1,2,5,5,5,5,5,5,1,2,2,4,5,2,0,0,0,1,5,0,1,1,5,0,0,5,4,2,3,3,5,3,3,2,4,4,4,4,5,5,5,2,1,5,3,3,5,5,5,1,5,3,2,1,5,5,5,5,1,5,5,3,4,4,3,5,5,0,3,5,5,0,0,2,1,4,4,0,2,5,3,1,4,5,1,1,1,0,1,5,1,1,1,1,1,0,5,1,5,5,1,1,1,1,5,4,4,1,1,4,1,3,3,4,1,1,5,5,5,5,4,5,5,5,4,4,5,5,4,5,1,1,5,5,1,1,1,1,0,5,1,3,5,5,1,1,5,1,1,5,1,5,1,0,5,5,1,1,1,1,1,1,1,3,5,3,3,4,3,4,5,5,4,5,5,1,5,5,5,4,5,5,1,5,4,1,1,5,1,1,1,5,5,5,5,4,5,4,1,5,5,5,5,1,0,1,1,1,4,1,1,1,1,1,5,1,5,4,1,1,3,1,5,4,1,4,1,4,1,1,1,4,1,3,5,5,5,5,5,5,5,1,5,5,1,5,5,2,1,5,4,1,1,1,1,1,1,1,1,5,5,3,5,5,1,5,1,1,1,5,5,1,5,1,0,1,4,4,2,1,5,1,0,0,5,1,1,1,5,5,5,1,5,1,4,1,5,5,3,5,1,1,5,5,0,5,0,1,5,3,5,5,5,5,1,5,5,3,5,3,5,1,5,1,3,0,3,5,1,1,2,5,2,2,5,5,2,5,1,1,1,5,0,1,5,1,5,1,1,0,5,1,0,3,3,4,1,1,1,1,1,1,1,2,5,5,1,1,3,1,5,1,1,5,1,1,5,2,1,1,1,5,1,1,1,1,1,1,5,1,1,1,1,1,1,1,1,5,1,1,1,1,5,4,5,1,1,1,4,5,1,1,4,5,4,1,3,2,5,1,1,1,2,1,1,1,1,1,5,1,1,5,5,1,5,0,5,5,1,5,5,1,5,5,5,5,5,5,4,5,5,5,5,5,4,1,4,3,5,4,3,3,3,0,5,5,5,5,1,5,1,5,5,5,0,1,3,5,5,1,1,5,1,5,1,5,1,1,5,5,5,1,2,1,2,4,5,3,5,1,5,1,1,5,5,5,0,1,1,5,1,2,2,1,4,1,5,5,1,1,5,1,1,5,1,1,1,1,1,1,1,1,1,1,4,5,2,1,2,3,1,0,1,1,5,5,5,5,5,5,1,5,5,5,5,1,1,1,1,1,5,5,5,4,5,1,1,5,5,3,1,1,4,4,1,2,1,5,5,5,1,1,1,5]

y1 = [1 if x > 2 else 0 for x in llm_data]
y2 = [1 if x > 2 else 0 for x in rater]
scores_k = cohen_kappa_score(llm_data, rater)
binary_scores_k = cohen_kappa_score(y1, y2)

print("Scores K", scores_k)
print("Binarises K", binary_scores_k)



## Run process over all samples in 10-cluster setting

In [ ]:
def get_llm_score(cluster_item, cluster_llm_title, cluster_llm_description):
    prompt_cluster = f"""Your task is to decide wether a sentence belong to a group, given the group title and its description.
    Give a score from 1 to 5 to judge if the sentence belongs to the group, where 1 means that the sentence is not belonging to the set, while 5 means that it definitely belongs to it.

    Provide in output a JSON with the following structure, do not output anything else.

    class Score(BaseModel):
        score: int # The score indicating if the sentence belongs to the group identified by title and description. The score can range from 1 to 5. Here 1 means that the sentence is not belonging to the set, while 5 means that it definitely belongs to it.

    Output:
        Score # the score assigned to the sentence
    """
    text_prompt = f"""
    Here is the group title:
    {cluster_llm_title}
    Here is the group description:
    {cluster_llm_description}

    Here is the sentence:
    {cluster_item}"""
    r = model_json(prompt_cluster, text_prompt)
    try:
        if r:
            return r['score']
        else:
            return None
    except:
        return None

In [ ]:
import os
clusterized_proposals = json.load(open(clusterized_proposals_fp))
cluster_summaries = json.load(open(full_summaries_fp))
df_result = None
if os.path.exists(f"{DATA_OUT_BASE_PATH}6.all_data_llm-judge.xlsx"):
    df_result = pd.read_excel(f"{DATA_OUT_BASE_PATH}6.all_data_llm-judge.xlsx")

cluster_samples_clean = {}
cluster_samples_full = {}
annotation_lines = []
for cluster_name, cluster_proposals in clusterized_proposals.items():
    cluster_llm_title = cluster_summaries[cluster_name]['title']
    cluster_llm_description = cluster_summaries[cluster_name]['description']
    print(cluster_name)
    print("\t", cluster_llm_title)
    print("\t", cluster_llm_description)

    cluster_samples = clusterized_proposals[cluster_name]
    non_cluster_samples = [{'cluster': other_cluster_name, 'item': x} for other_cluster_name in clusterized_proposals.keys() if other_cluster_name != cluster_name for x in clusterized_proposals[other_cluster_name]]

    #cluster_samples = cluster_samples[:3]
    #non_cluster_samples = non_cluster_samples[:3]
    for cluster_item in tqdm(cluster_samples, desc="Processing cluster items"):
        if df_result and len(df_result[(df_result['id']==cluster_item['id']) & (df_result['id']==cluster_llm_title)]) > 0:
            score = df_result[(df_result['id']==cluster_item['id']) & (df_result['id']==cluster_llm_title)]['Q3'].tolist()[0]
        else:
            score = get_llm_score(cluster_item['title'], cluster_llm_title, cluster_llm_description)
        if score:
            annotation_lines.append({
                'llm_title': cluster_llm_title,
                'llm_description': cluster_llm_description,
                'id':cluster_item['id'],
                'title':cluster_item['title'],
                'language':cluster_item['language'],
                'Q3': score,
                'Belong to cluster': True,
                'Orginal cluster': cluster_name

            })

    for non_cluster_item in tqdm(non_cluster_samples, desc="Processing NON cluster items"):
        if df_result and len(df_result[(df_result['id']==non_cluster_item['id']) & (df_result['id']==cluster_llm_title)]) > 0:
            score = df_result[(df_result['id']==non_cluster_item['id']) & (df_result['id']==cluster_llm_title)]['Q3'].tolist()[0]
        else:
            score = get_llm_score(non_cluster_item['item']['title'], cluster_llm_title, cluster_llm_description)
        if score:
            annotation_lines.append({
                'llm_title': cluster_llm_title,
                'llm_description': cluster_llm_description,
                'id':non_cluster_item['item']['id'],
                'title':non_cluster_item['item']['title'],
                'language':non_cluster_item['item']['language'],
                'Q3': score,
                'Belong to cluster': False,
                'Orginal cluster': non_cluster_item['cluster']
            })
    pd.DataFrame(annotation_lines).to_excel(f"{DATA_OUT_BASE_PATH}6.all_data_llm-judge.xlsx", index=False)
pd.DataFrame(annotation_lines).to_excel(f"{DATA_OUT_BASE_PATH}6.all_data_llm-judge.xlsx", index=False)